# 🚨 Tool Pengambilan Data SPOOF & Auto-Import REAL (MiniFASNet)

Notebook ini didesain khusus sesuai kebutuhan Anda:
1. **Foto REAL (Wajah Asli)**: **Otomatis diimpor dan di-crop** dari folder enrollment yang sudah ada (`faza/`, `raffin/`, `rafy/`). Anda **tidak perlu** foto wajah asli lagi dari awal!
2. **Foto SPOOF (Wajah Palsu)**: Diambil secara interaktif dari webcam laptop dengan mengarahkan layar HP, tablet, atau kertas foto ke depan kamera.

---

### 🎯 Variasi Serangan SPOOF yang Sangat Disarankan:
* **Screen Attack (Layar HP / Tablet)**:
  * Tampilkan foto wajah di HP, arahkan ke webcam laptop.
  * Variasikan kecerahan layar HP (terang & redup).
  * Miringkan HP sedikit (kiri/kanan/atas/bawah) untuk menangkap pantulan cahaya dan garis piksel (*moiré pattern*).
  * Gerakkan HP mendekat (20cm) dan menjauh (60cm).
* **Video Replay Attack**:
  * Putar video orang berbicara atau kedip di HP di depan webcam.
* **Print Attack (Kertas)**:
  * Pegang foto hasil print kertas atau foto cetak di depan webcam.

---

### ⌨️ Kontrol Cepat saat Kamera Aktif:
| Tombol | Aksi |
| :--- | :--- |
| **`[SPASI]`** atau **`[S]`** | Ambil 1 foto **SPOOF** |
| **`[B]`** | **Burst Capture (15 Frame)**: Ambil 15 foto beruntun sambil Anda gerakkan HP perlahan |
| **`[C]`** | Ganti Jenis Spoof (`screen_hp`, `paper_print`, `video_replay`, `screen_angle`) |
| **`[R]`** | Ambil foto **REAL** manual (opsional jika ingin tambah wajah asli dari webcam) |
| **`[Q]`** atau **`[ESC]`** | Selesai & Keluar |

In [ ]:
# Cell 1: Setup Library & Folder Dataset
import os
import sys
import cv2
import time
import datetime
import csv
import glob
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Konfigurasi Direktori Dataset
DATASET_ROOT = Path("dataset_fas")
DIRS = {
    "real_raw": DATASET_ROOT / "real" / "raw",
    "real_crop": DATASET_ROOT / "real" / "crop_2.7x",
    "spoof_raw": DATASET_ROOT / "spoof" / "raw",
    "spoof_crop": DATASET_ROOT / "spoof" / "crop_2.7x",
}

for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATASET_ROOT / "annotations.csv"
if not CSV_PATH.exists():
    with open(CSV_PATH, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "filename", "label", "category", "sub_type", "bbox_x", "bbox_y", "bbox_w", "bbox_h"])

print("✅ Struktur folder dataset siap di:", DATASET_ROOT.resolve())

In [ ]:
# Cell 2: Modul Cropper MiniFASNet (Skala 2.7x -> 80x80) & Detektor YuNet

class MiniFASCropper:
    """
    Cropper khusus arsitektur MiniFASNet.
    Memperbesar bounding box wajah 2.7x untuk menyertakan bingkai HP, kertas, & latar.
    """
    @staticmethod
    def get_expanded_box(src_w, src_h, bbox, scale=2.7):
        x, y, bw, bh = bbox
        if bh > 0 and bw > 0:
            scale = min((src_h - 1) / bh, min((src_w - 1) / bw, scale))
        else:
            scale = 1.0

        new_w = bw * scale
        new_h = bh * scale
        cx = x + bw / 2.0
        cy = y + bh / 2.0

        left = cx - new_w / 2.0
        top = cy - new_h / 2.0
        right = cx + new_w / 2.0
        bottom = cy + new_h / 2.0

        if left < 0:
            right -= left
            left = 0
        if top < 0:
            bottom -= top
            top = 0
        if right > src_w - 1:
            left -= (right - src_w + 1)
            right = src_w - 1
        if bottom > src_h - 1:
            top -= (bottom - src_h + 1)
            bottom = src_h - 1

        return max(0, int(left)), max(0, int(top)), min(src_w - 1, int(right)), min(src_h - 1, int(bottom))

    @classmethod
    def crop(cls, img, bbox, scale=2.7, out_size=(80, 80)):
        src_h, src_w = img.shape[:2]
        l, t, r, b = cls.get_expanded_box(src_w, src_h, bbox, scale=scale)
        patch = img[t : b + 1, l : r + 1]
        if patch.size == 0:
            return cv2.resize(img, out_size)
        return cv2.resize(patch, out_size, interpolation=cv2.INTER_LINEAR)


# Inisialisasi Detektor YuNet (dengan fallback Haar)
YUNET_PATHS = [
    Path("face_detection_yunet_2023mar.onnx"),
    Path("backend_web/face_detection_yunet_2023mar.onnx"),
    Path("../face_detection_yunet_2023mar.onnx"),
    Path("../fr_experiment/models/face_detection_yunet_2023mar.onnx"),
]
yunet_path = next((p for p in YUNET_PATHS if p.exists()), None)
detector_type = "none"
yunet_detector = None
haar_detector = None

if yunet_path:
    try:
        yunet_detector = cv2.FaceDetectorYN.create(str(yunet_path), "", (320, 320), 0.45, 0.3, 5000)
        detector_type = "yunet"
        print(f"✅ Detektor Wajah: YuNet ONNX ({yunet_path})")
    except Exception as e:
        print(f"⚠️ YuNet gagal ({e}), beralih ke Haar.")

if detector_type == "none":
    cascade_file = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    haar_detector = cv2.CascadeClassifier(cascade_file)
    detector_type = "haar"
    print("✅ Detektor Wajah: Haar Cascade")

def detect_primary_face(frame):
    h, w = frame.shape[:2]
    if detector_type == "yunet":
        yunet_detector.setInputSize((w, h))
        _, faces = yunet_detector.detect(frame)
        if faces is not None and len(faces) > 0:
            best = max(faces, key=lambda f: f[2] * f[3])
            x, y, bw, bh = int(best[0]), int(best[1]), int(best[2]), int(best[3])
            x, y = max(0, x), max(0, y)
            bw, bh = min(w - x, bw), min(h - y, bh)
            return (x, y, bw, bh)
    else:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = haar_detector.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=5, minSize=(50, 50))
        if len(faces) > 0:
            best = max(faces, key=lambda f: f[2] * f[3])
            return tuple(int(v) for v in best)
    return None

In [ ]:
# Cell 3: AUTO-IMPORT & PREPROCESS FOTO REAL DARI ENROLLMENT DATASET
# Jalankan sel ini untuk otomatis mengonversi foto wajah asli yang sudah Anda miliki!

ENROLLMENT_SOURCES = [
    Path("faza"), Path("raffin"), Path("rafy"),
    Path("../faza"), Path("../raffin"), Path("../rafy"),
    Path("dataset"), Path("../dataset"),
]

valid_sources = [p for p in ENROLLMENT_SOURCES if p.exists() and p.is_dir()]
print(f"🔍 Menemukan {len(valid_sources)} folder dataset enrollment:", [str(p) for p in valid_sources])

imported_count = 0
skipped_count = 0

for folder in valid_sources:
    image_files = list(folder.glob("*.jpg")) + list(folder.glob("*.jpeg")) + list(folder.glob("*.png"))
    print(f"   📂 Memproses '{folder.name}': {len(image_files)} foto...")
    for img_path in image_files:
        img = cv2.imread(str(img_path))
        if img is None:
            skipped_count += 1
            continue

        # Resize jika terlalu raksasa agar deteksi akurat
        h, w = img.shape[:2]
        if max(h, w) > 1280:
            scale_f = 1280 / max(h, w)
            img = cv2.resize(img, (int(w * scale_f), int(h * scale_f)))

        bbox = detect_primary_face(img)
        if bbox is None:
            # Fallback jika wajah memenuhi layar
            bbox = (0, 0, img.shape[1], img.shape[0])

        dest_name = f"real_imported_{folder.name}_{img_path.stem}.jpg"
        
        # Simpan Raw Frame
        raw_dest = DIRS["real_raw"] / dest_name
        cv2.imwrite(str(raw_dest), img)

        # Simpan Crop 2.7x (80x80) patch MiniFASNet
        crop_patch = MiniFASCropper.crop(img, bbox, scale=2.7, out_size=(80, 80))
        crop_dest = DIRS["real_crop"] / dest_name
        cv2.imwrite(str(crop_dest), crop_patch)

        imported_count += 1

total_real_now = len(list(DIRS["real_crop"].glob("*.jpg")))
print(f"\n🎉 SUKSES! {imported_count} foto wajah REAL berhasil diimpor & dipotong (80x80)!")
print(f"📊 Total data REAL yang siap digunakan: {total_real_now} sampel")
print(f"💡 Sekarang Anda HANYA PERLU mengambil data SPOOF pada Cell berikutnya agar seimbang!")

In [ ]:
# Cell 4: WEBCAM CAPTURE KHUSUS DATA SPOOF (WAJAH PALSU / LAYAR HP / KERTAS)
# Tekan [SPASI] atau [S] untuk ambil 1 foto spoof.
# Tekan [B] untuk burst 15 foto beruntun sambil menggerakkan layar HP perlahan.
# Tekan [Q] untuk selesai.

def run_spoof_capture_tool(camera_index=0):
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        print(f"❌ Gagal membuka kamera {camera_index}. Coba camera_index=1 jika pakai webcam eksternal.")
        return

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    spoof_types = ["screen_hp", "screen_hp_angle", "paper_print", "video_replay", "screen_tablet"]
    current_type_idx = 0

    count_real = len(list(DIRS["real_crop"].glob("*.jpg")))
    count_spoof = len(list(DIRS["spoof_crop"].glob("*.jpg")))

    burst_mode = False
    burst_remaining = 0
    last_burst_time = 0.0

    flash_timer = 0
    flash_color = (0, 0, 255)

    window_name = "MiniFASNet SPOOF Collector (Tekan [Q] untuk Keluar)"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, 1000, 620)

    print("\n🎥 Jendela kamera aktif!")
    print("   Arahkan layar HP / Kertas Foto ke arah webcam.")
    print("   [SPASI] / [S] -> Ambil 1 Foto SPOOF")
    print("   [B]           -> BURST 15 Foto (Gerakkan HP perlahan agar banyak sudut)")
    print("   [C]           -> Ganti Jenis Serangan Spoof")
    print("   [Q]           -> Selesai\n")

    def save_spoof_sample(frame, bbox, sub_type):
        nonlocal count_spoof
        now_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:19]
        file_base = f"spoof_{sub_type}_{now_str}.jpg"

        # 1. Simpan Raw
        cv2.imwrite(str(DIRS["spoof_raw"] / file_base), frame)

        # 2. Simpan Crop Patch 2.7x (80x80)
        crop_patch = MiniFASCropper.crop(frame, bbox, scale=2.7, out_size=(80, 80))
        cv2.imwrite(str(DIRS["spoof_crop"] / file_base), crop_patch)

        # 3. Log CSV
        with open(CSV_PATH, mode="a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                datetime.datetime.now().isoformat(),
                file_base,
                1,
                "spoof",
                sub_type,
                bbox[0], bbox[1], bbox[2], bbox[3]
            ])
        count_spoof += 1

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)  # mirror
            display_frame = frame.copy()
            h, w = frame.shape[:2]

            bbox = detect_primary_face(frame)

            if bbox is not None:
                bx, by, bw, bh = bbox
                # Bounding box deteksi wajah (merah karena target spoof)
                cv2.rectangle(display_frame, (bx, by), (bx + bw, by + bh), (0, 0, 255), 2)

                # Area konteks 2.7x (kuning)
                l, t, r, b = MiniFASCropper.get_expanded_box(w, h, bbox, scale=2.7)
                cv2.rectangle(display_frame, (l, t), (r, b), (0, 220, 255), 1)
                cv2.putText(display_frame, "Area Konteks Layar/Kertas (2.7x)", (l, max(18, t - 6)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 220, 255), 1)
            else:
                cv2.putText(display_frame, "Wajah di Layar HP Tidak Terdeteksi!", (25, 85),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                cv2.putText(display_frame, "Dekatkan layar HP atau naikkan kecerahan layar.", (25, 115),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

            # Burst Mode Auto-Save
            if burst_mode and burst_remaining > 0 and bbox is not None:
                now_t = time.time()
                if now_t - last_burst_time >= 0.18:  # 180ms interval
                    save_spoof_sample(frame, bbox, spoof_types[current_type_idx])
                    burst_remaining -= 1
                    last_burst_time = now_t
                    flash_timer = 2
                    if burst_remaining == 0:
                        burst_mode = False
                        print(f"✅ Burst selesai! Total SPOOF sekarang: {count_spoof}")

            # Efek Flash Visual merah saat simpan spoof
            if flash_timer > 0:
                cv2.rectangle(display_frame, (0, 0), (w, h), (0, 0, 255), 10)
                flash_timer -= 1

            # Overlay HUD Panel
            hud = display_frame.copy()
            cv2.rectangle(hud, (0, 0), (w, 60), (15, 15, 15), -1)
            cv2.rectangle(hud, (0, h - 50), (w, h), (15, 15, 15), -1)
            display_frame = cv2.addWeighted(hud, 0.7, display_frame, 0.3, 0)

            # Info Bar Atas
            cv2.putText(display_frame, f"REAL (Enrollment): {count_real}", (25, 38),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 100), 2)
            cv2.putText(display_frame, f"SPOOF TERKUMPUL: {count_spoof}", (360, 38),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.85, (0, 80, 255), 2)
            
            # Target Balance Indicator
            balance_txt = "SEIMBANG ✓" if abs(count_real - count_spoof) < 15 else f"Target: butuh {max(0, count_real - count_spoof)} lagi"
            cv2.putText(display_frame, f"Status: {balance_txt}", (w - 280, 38),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 1)

            # Info Bar Bawah
            if burst_mode:
                b_msg = f"🔥 BURST MENGAMBIL SPOOF... Sisa {burst_remaining} frame! Gerakkan HP sedikit!"
                cv2.putText(display_frame, b_msg, (25, h - 18),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)
            else:
                guide_msg = f"[SPASI/S] Simpan SPOOF  |  [B] Burst 15x  |  [C] Tipe: <{spoof_types[current_type_idx]}>  |  [Q] Keluar"
                cv2.putText(display_frame, guide_msg, (25, h - 18),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

            cv2.imshow(window_name, display_frame)
            key = cv2.waitKey(1) & 0xFF

            if key in [ord('q'), 27]:
                break
            elif key in [32, ord('s'), ord('S')]:  # SPASI atau 'S'
                if bbox is not None:
                    save_spoof_sample(frame, bbox, spoof_types[current_type_idx])
                    flash_timer = 4
                    print(f"🚨 Disimpan SPOOF #{count_spoof} ({spoof_types[current_type_idx]})")
                else:
                    print("⚠️ Pastikan wajah pada layar HP berada dalam frame kamera!")
            elif key in [ord('b'), ord('B')]:
                burst_mode = True
                burst_remaining = 15
                print(f"⚡ BURST 15 SPOOF DIMULAI! Gerakkan layar HP sedikit miring & dekat/jauh...")
            elif key in [ord('c'), ord('C')]:
                current_type_idx = (current_type_idx + 1) % len(spoof_types)
                print(f"🔄 Tipe serangan spoof: {spoof_types[current_type_idx]}")
            elif key in [ord('r'), ord('R')]:
                # Opsional: jika ingin ambil foto real tambahan langsung dari webcam
                if bbox is not None:
                    now_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:19]
                    cv2.imwrite(str(DIRS["real_raw"] / f"real_cam_{now_str}.jpg"), frame)
                    crop_patch = MiniFASCropper.crop(frame, bbox, scale=2.7, out_size=(80, 80))
                    cv2.imwrite(str(DIRS["real_crop"] / f"real_cam_{now_str}.jpg"), crop_patch)
                    count_real += 1
                    print(f"📸 Disimpan REAL #{count_real}")

    finally:
        cap.release()
        cv2.destroyAllWindows()
        print(f"\n🎉 SELESAI PENGAMBILAN DATA SPOOF!")
        print(f"   - Total Real (Enrollment): {count_real}")
        print(f"   - Total Spoof Terkumpul : {count_spoof}")

# Jalankan pengambil data spoof:
run_spoof_capture_tool(camera_index=0)

In [ ]:
# Cell 5: Review & Visualisasi Sampel Real (Enrollment) vs Spoof (Layar HP/Kertas)

real_crops = sorted(list(DIRS["real_crop"].glob("*.jpg")))
spoof_crops = sorted(list(DIRS["spoof_crop"].glob("*.jpg")))

print(f"📊 RINGKASAN DATASET FAS:")
print(f"   - REAL (Enrollment Asli) : {len(real_crops)} sampel")
print(f"   - SPOOF (Layar HP/Kertas): {len(spoof_crops)} sampel")
print(f"   - Total Keseluruhan      : {len(real_crops) + len(spoof_crops)} sampel")

# Perbandingan visual: 5 Real dari enrollment vs 5 Spoof dari layar HP
n_samples = min(5, max(len(real_crops), len(spoof_crops), 1))
fig, axes = plt.subplots(2, n_samples, figsize=(3 * n_samples, 6))

for i in range(n_samples):
    # Baris 0: REAL (Dari dataset enrollment)
    ax_r = axes[0, i] if n_samples > 1 else axes[0]
    if i < len(real_crops):
        img = cv2.imread(str(real_crops[-(i + 1)]))
        ax_r.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax_r.set_title(f"Real (Enrollment) #{i+1}", color="green", fontsize=10)
    else:
        ax_r.text(0.5, 0.5, "Belum ada", ha="center", va="center")
    ax_r.axis("off")

    # Baris 1: SPOOF (Dari HP/Kertas)
    ax_s = axes[1, i] if n_samples > 1 else axes[1]
    if i < len(spoof_crops):
        img = cv2.imread(str(spoof_crops[-(i + 1)]))
        ax_s.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax_s.set_title(f"Spoof (Layar HP) #{i+1}", color="red", fontsize=10)
    else:
        ax_s.text(0.5, 0.5, "Belum ada", ha="center", va="center")
    ax_s.axis("off")

plt.suptitle("Perbandingan Patch MiniFASNet (80x80 Scale 2.7x): REAL vs SPOOF", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Otomatis Split Train (80%) dan Validation (20%) untuk Training PyTorch

def prepare_train_val_split(val_ratio=0.2):
    SPLIT_ROOT = Path("dataset_fas_split")
    for split in ["train", "val"]:
        for cls_name in ["0_real", "1_spoof"]:
            (SPLIT_ROOT / split / cls_name).mkdir(parents=True, exist_ok=True)

    def split_and_copy(src_files, cls_folder):
        files = list(src_files)
        random.seed(42)
        random.shuffle(files)
        val_count = int(len(files) * val_ratio)
        val_files = files[:val_count]
        train_files = files[val_count:]

        for f in train_files:
            shutil.copy2(f, SPLIT_ROOT / "train" / cls_folder / f.name)
        for f in val_files:
            shutil.copy2(f, SPLIT_ROOT / "val" / cls_folder / f.name)
        return len(train_files), len(val_files)

    r_tr, r_val = split_and_copy(DIRS["real_crop"].glob("*.jpg"), "0_real")
    s_tr, s_val = split_and_copy(DIRS["spoof_crop"].glob("*.jpg"), "1_spoof")

    print("🎉 DATASET SIAP DILATIH!")
    print(f"📁 Folder: {SPLIT_ROOT.resolve()}")
    print(f"   - TRAIN: Real = {r_tr}, Spoof = {s_tr} (Total = {r_tr + s_tr})")
    print(f"   - VAL  : Real = {r_val}, Spoof = {s_val} (Total = {r_val + s_val})")

if len(real_crops) >= 5 and len(spoof_crops) >= 5:
    prepare_train_val_split(val_ratio=0.2)
else:
    print(f"💡 Ambil minimal 5-10 foto SPOOF terlebih dahulu (saat ini Spoof = {len(spoof_crops)}).")